In [1]:
import os 

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio'

Entity

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareModelConfig:
    root_dir: Path
    model_path: Path
    params_image_size: list
    params_classes: int

In [6]:
from pathlib import Path

from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories


In [7]:

class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_model_config(self) -> PrepareModelConfig:

        config = self.config.prepare_model

        create_directories([config.root_dir])

        prepare_model_config = PrepareModelConfig(
            root_dir=Path(config.root_dir),
            model_path=Path(config.model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_classes=self.params.CLASSES,
        )

        return prepare_model_config

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    BatchNormalization,
    MaxPooling2D,
    Dropout,
    Flatten,
    Dense,
)
from tensorflow.keras.regularizers import l2
from pathlib import Path

In [9]:


class PrepareModel:

    def __init__(self, config):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def create_model(self):

        weight_decay = 0.0001

        model = Sequential()

        # Block 1
        model.add(
            Conv2D(
                32,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
                input_shape=tuple(self.config.params_image_size),
            )
        )
        model.add(BatchNormalization())

        model.add(
            Conv2D(
                32,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(MaxPooling2D((2, 2)))
        model.add(Dropout(0.2))

        # Block 2
        model.add(
            Conv2D(
                64,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(
            Conv2D(
                64,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(MaxPooling2D((2, 2)))
        model.add(Dropout(0.3))

        # Block 3
        model.add(
            Conv2D(
                128,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(
            Conv2D(
                128,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(MaxPooling2D((2, 2)))
        model.add(Dropout(0.4))

        # Block 4
        model.add(
            Conv2D(
                256,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(
            Conv2D(
                256,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=l2(weight_decay),
            )
        )
        model.add(BatchNormalization())

        model.add(MaxPooling2D((2, 2)))
        model.add(Dropout(0.5))

        # Classification Head
        model.add(Flatten())
        model.add(Dense(self.config.params_classes, activation="softmax"))

        model.summary()

        self.save_model(self.config.model_path, model)

In [10]:



try:

    config = ConfigurationManager()

    prepare_model_config = config.get_prepare_model_config()

    prepare_model = PrepareModel(prepare_model_config)

    prepare_model.create_model()

    print("Model created successfully.")

except Exception as e:
    raise e

[2026-07-01 19:41:10,556: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 19:41:10,594: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 19:41:10,599: INFO: common: created directory at: artifacts]
[2026-07-01 19:41:10,599: INFO: common: created directory at: artifacts/prepare_model]
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 batch_normalization (Batch  (None, 32, 32, 32)        128       
 Normalization)                                                  
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 32)        9248      
                                                                 
 batch_normalization_1 (Bat  (None, 32, 32, 32)    